# Notebook 05b — ClinicalBERT Tokenization Analysis

---

## Objectives

Decide `max_length` (the fixed sequence length used for padding/truncation in Notebook 06b) by inspecting how many BERT subword tokens each cleaned report produces under the `emilyalsentzer/Bio_ClinicalBERT` tokenizer.

> 🔧 **Fix — data leakage removed:** the original version of this notebook computed token-length statistics on `train + validation + test` combined, then picked `max_length` from that combined distribution — using test-set text to inform a preprocessing decision before any model ever sees the test set. This version computes the statistics and the `max_length` decision from **train only** (Section 4). Validation and test are then checked **separately, only as a verification** that the train-derived `max_length` also covers them adequately (Section 6) — their numbers never feed back into the decision itself.


## 1. Load Split Reports


In [2]:
DATASET_DIR = Path("/kaggle/input/datasets/mariammohamed1095/reportssss/datasets/processed/nlp")

train_df = pd.read_csv(DATASET_DIR / "train_reports_clean.csv")
val_df   = pd.read_csv(DATASET_DIR / "validation_reports_clean.csv")
test_df  = pd.read_csv(DATASET_DIR / "test_reports_clean.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(257, 6)
(56, 6)
(56, 6)


## 2. Tokenizer


In [4]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(tokenizer)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='emilyalsentzer/Bio_ClinicalBERT', vocab_size=28996, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


## 3. Restrict to TRAIN ONLY

🔧 **Fix:** previously this cell built `all_reports` from `pd.concat([train_df, val_df, test_df])`. The `max_length` decision below must only ever see training data — validation and test are held out completely until Section 6, where they are checked but not used to choose anything.


In [ ]:
train_reports = train_df  # 🔧 FIX: train only, not train+val+test

print(len(train_reports))


## 4. Token Length Distribution (Train Only)


In [ ]:
token_lengths = []

for report in train_reports["clean_report"]:

    tokens = tokenizer.encode(
        report,
        add_special_tokens=True,
        truncation=False
    )

    token_lengths.append(len(tokens))

token_lengths = np.array(token_lengths)


### 4.1 Summary Statistics (Train Only)


In [6]:
stats = pd.DataFrame({

    "Metric":[

        "Minimum",

        "Maximum",

        "Mean",

        "Median"

    ],

    "Value":[

        token_lengths.min(),

        token_lengths.max(),

        token_lengths.mean(),

        np.median(token_lengths)

    ]

})

stats

,Metric,Value
0,Minimum,78.000000
1,Maximum,203.000000
2,Mean,134.482385
3,Median,133.000000


### 4.2 Percentiles (Train Only)


In [7]:
percentiles = pd.DataFrame({

    "Percentile":[

        50,

        75,

        90,

        95,

        99

    ],

    "Length":[

        np.percentile(token_lengths,50),

        np.percentile(token_lengths,75),

        np.percentile(token_lengths,90),

        np.percentile(token_lengths,95),

        np.percentile(token_lengths,99),

    ]

})

percentiles

,Percentile,Length
0,50,133.00
1,75,148.00
2,90,165.00
3,95,173.60
4,99,190.32


## 5. Truncation Tradeoff by Candidate `max_length` (Train Only)


In [8]:
limits = [128,160,192,256]

rows = []

for limit in limits:

    truncated = np.sum(
        token_lengths > limit
    )

    rows.append({

        "Max Length": limit,

        "Reports Truncated": truncated,

        "Percentage": truncated / len(token_lengths) * 100

    })

truncation_df = pd.DataFrame(rows)

truncation_df

,Max Length,Reports Truncated,Percentage
0,128,214,57.994580
1,160,49,13.279133
2,192,3,0.813008
3,256,0,0.000000


### 5.1 Decision


In [9]:
recommended = 256

print("="*60)

print("Recommended max_length :", recommended)

print()

print("Longest report :", token_lengths.max())

print("Reports :", len(token_lengths))

Recommended max_length : 256

Longest report : 203
Reports : 369


## 6. Verification on Validation & Test (Not Used to Choose `max_length`)

Reports the same truncation statistics on validation and test **separately**, purely to confirm the train-derived `max_length` generalizes. If either split showed much heavier truncation than train here, that would be a sign of distribution shift worth investigating — but it would still not be a reason to widen `max_length` using test-set information; the fix would be to revisit train-set coverage or accept the truncation rate.


In [ ]:
def truncation_report(df, name, max_length=recommended):

    lengths = np.array([
        len(tokenizer.encode(r, add_special_tokens=True, truncation=False))
        for r in df["clean_report"]
    ])

    truncated = int(np.sum(lengths > max_length))

    return {
        "Split": name,
        "Reports": len(lengths),
        "Max Tokens": int(lengths.max()),
        "Truncated at max_length": truncated,
        "Truncated %": round(truncated / len(lengths) * 100, 2),
    }

verification_df = pd.DataFrame([
    truncation_report(train_reports, "Train"),
    truncation_report(val_df, "Validation"),
    truncation_report(test_df, "Test"),
])

verification_df


## 7. Save Results


In [ ]:
REPORT_DIR = Path("/kaggle/working/reports/nlp/results")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

stats.to_csv(
    REPORT_DIR / "token_ClinicalBERT_statistics.csv",
    index=False
)

percentiles.to_csv(
    REPORT_DIR / "token_ClinicalBERT_percentiles.csv",
    index=False
)

truncation_df.to_csv(
    REPORT_DIR / "token_ClinicalBERT_truncation_analysis.csv",
    index=False
)

verification_df.to_csv(
    REPORT_DIR / "token_split_verificationb.csv",
    index=False
)

print("Tokenization analysis saved.")